In [ ]:
# Spanish Real Estate Forecasting - Data Model Training
# This notebook is dedicated to training a predictive model for forecasting real estate prices in Spain. 
# I will use the cleaned dataset to train a machine learning model, evaluate its performance, and make predictions.

# Importing necessary libraries for data manipulation, visualization, and machine learning
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler
from models import prophet_model




In [2]:
# Loading the cleaned dataset from the pickle file for analysis
BASE_DIR = Path.cwd().parent.parent
data_path = BASE_DIR / "data" / "cleaned_real_estate_data.pkl"
df = pd.read_pickle(data_path)
df.sample(5)

,ano,trim,geo,cod-ca,ca,cod-prv,prv,sector,variable,valor,tipo,metrica
147867,2007,2,comunidad,12,galicia,98,galicia,commercial,nav-imp,492197.00,nav,imp
21960,2007,2,nacional,99,espana,99,espana,residential,viv-pm2,1979.42,viv,pm2
36877,2021,1,provincia,2,aragon,50,zaragoza,residential,gar-num,3139.00,gar,num
37884,2024,3,provincia,2,aragon,44,teruel,residential,gar-num,356.00,gar,num
46754,2017,2,provincia,7,castilla y leon,24,leon,residential,gar-imp,8480.00,gar,imp


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82080 entries, 0 to 164159
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ano       82080 non-null  int64  
 1   trim      82080 non-null  int64  
 2   geo       82080 non-null  object 
 3   cod-ca    82080 non-null  int64  
 4   ca        82080 non-null  object 
 5   cod-prv   82080 non-null  int64  
 6   prv       82080 non-null  object 
 7   sector    82080 non-null  object 
 8   variable  82080 non-null  object 
 9   valor     82080 non-null  float64
 10  tipo      82080 non-null  object 
 11  metrica   82080 non-null  object 
dtypes: float64(1), int64(4), object(7)
memory usage: 8.1+ MB


*PREPARING THE DATA FOR FORECASTING*

In [4]:
# Converting categorical columns to the appropriate data type to optimize memory usage and improve performance during modeling.
cat_cols = [
    "geo",
    "ca",
    "prv",
    "sector",
    "variable",
    "tipo",
    "metrica"
]

for col in cat_cols:
    df[col] = df[col].astype("category")

# Specifically converting "cod-ca" and "cod-prv" to categorical data types as they represent categorical information 
# about the autonomous communities and provinces in Spain.
df["cod-ca"] = df["cod-ca"].astype("category")
df["cod-prv"] = df["cod-prv"].astype("category")

In [5]:
# Creating a datetime column from the "ano" and "trim" columns to facilitate time series analysis and modeling.
df["date"] = pd.PeriodIndex.from_fields(year=df["ano"], quarter=df["trim"], freq="Q").to_timestamp()
df = df.sort_values("date")

In [6]:
# Exploring the average property value over time to identify trends and seasonality in the real estate market.
ts = (
    df.groupby(
        ["date", "tipo"],
        observed = True)
        ["valor"]
        .mean()
        .reset_index()
)

In [7]:
# Applying log transformation to the average property values in the time series data 
# to reduce skewness and make the data more suitable for modeling. It helps to stabilize the variance 
# and improve the performance of machine learning algorithms.
ts["valor_log"] = np.log1p(ts["valor"])

In [8]:
# Extracting year and quarter from the date column to create additional features for time series analysis and modeling.
ts["year"] = ts["date"].dt.year
ts["quarter"] = ts["date"].dt.quarter
ts["quarter_sin"] = np.sin(2*np.pi*ts["quarter"]/4)
ts["quarter_cos"] = np.cos(2*np.pi*ts["quarter"]/4)

In [9]:
# Creating lag features to capture the temporal dependencies in the time series data, 
# which can help improve the performance of predictive models by providing information about past values.
ts["lag_1"] = ts["valor_log"].shift(1)
ts["lag_2"] = ts["valor_log"].shift(2)
ts["lag_4"] = ts["valor_log"].shift(4)

In [10]:
# Creating rolling mean and standard deviation features to capture the local trends and volatility in the time series data,
# which can help improve the performance of predictive models by providing information about recent patterns.
ts["rolling_mean_4"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(4).mean()))
ts["rolling_std_4"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(4).std()))
ts["rolling_mean_8"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(8).mean()))
ts["rolling_std_8"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(8).std()))
ts["diff_1"] = ts["valor_log"].diff(1)


C:\Users\carme\AppData\Local\Temp\ipykernel_56372\3860770452.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ts["rolling_mean_4"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(4).mean()))
C:\Users\carme\AppData\Local\Temp\ipykernel_56372\3860770452.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ts["rolling_std_4"] = (ts.groupby("tipo")["valor_log"].transform(lambda x: x.rolling(4).std()))
C:\Users\carme\AppData\Local\Temp\ipykernel_56372\3860770452.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observ

In [11]:
# Dropping rows with missing values that were introduced by the lag and rolling features,
# which are necessary for training the predictive model without errors.
ts = ts.dropna()

In [12]:
# Dividing the time series data into training and testing sets based on a specific date 
# to evaluate the performance of the predictive model on unseen data.
train = ts[ts["date"] < "2023-01-01"]
test = ts[ts["date"] >= "2023-01-01"]

In [13]:
# Encoding the "tipo" categorical variable using one-hot encoding to convert it into a format that can be used by machine learning algorithms,
# while dropping the first category to avoid multicollinearity.
train = pd.get_dummies(train, columns=["tipo"], drop_first=True)
test = pd.get_dummies(test, columns=["tipo"], drop_first=True)

# Aligning the training and testing datasets to ensure they have the same columns after one-hot encoding,
# filling any missing columns with zeros to maintain consistency for model training and evaluation.
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [14]:
# Scaling the features using StandardScaler to normalize the data by ensuring that all features are on a similar scale.
scaler = StandardScaler()
feature_cols = features = [
"lag_1", 
"lag_2",   
"lag_4",    
"rolling_mean_4",    
"rolling_std_4",
"rolling_mean_8",
"rolling_std_8",    
"quarter_sin",    
"quarter_cos"]
target_col = "valor_log" # The target variable for modeling is the log-transformed average property value.
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# Preparing the training and testing datasets by separating the features and target variable for model training and evaluation.
X_train = train.drop(columns=["valor", "valor_log", "date"])
y_train = train[target_col]
X_test = test.drop(columns=["valor", "valor_log", "date"])
y_test = test[target_col]

In [ ]:
prophet_model = prophet_model.ProphetModel()
prophet_model.fit(train)
predictions = prophet_model.predict(periods = len(test))
mae = prophet_model.evaluate(test)
print(f"Mean Absolute Error: {mae}")

In [ ]:
plt.figure(figsize=(10, 6))
prophet_model.model.plot(predictions, xlabel="Date", ylabel="Log of Average Property Value", title="Prophet Model Predictions vs Actual Values")
plt.show()



In [ ]:
plt.figure(figsize=(10, 6))
prophet_model.model.plot_components(predictions, xlabel="Date", ylabel="Log of Average Property Value", title="Prophet Model Components")
plt.show()